In [18]:
import pandas as pd
import re

df = pd.read_csv('data/movies_2024.csv')

In [19]:
df.shape

(2455, 2)

In [20]:
df.columns

Index(['Movie_Name', 'Storyline'], dtype='str')

In [21]:
df.head(3)

,Movie_Name,Storyline
0,Dune: Part Two,"Paul Atreides unites with the Fremen while on a warpath of revenge against the conspirators who destroyed his family. Facing a choice between the love of his life and the fate of the universe, he endeavors to prevent a terrible future."
1,The Life of Chuck,"A life-affirming, genre-bending story about three chapters in the life of an ordinary man named Charles Krantz."
2,The Substance,"A fading celebrity takes a black-market drug: a cell-replicating substance that helps her create a younger, better version of herself."


In [22]:
df.tail(3)

,Movie_Name,Storyline
2452,"April, Come She Will","On the morning following the eve of their engagement, the fiancée disappears from the man's side. He is left to reminisce about why she chose to run away."
2453,Sci-Fi Vixens from Beyond,"A curious alien researcher arrives on Earth to study human behavior, becoming fascinated by the attractive space warriors and cosmic adventurers who protect our planet from otherworldly threats."
2454,Aïcha,Explores how far one can go to break free from their past.


Text Cleaning

In [27]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# a Try-Catch check point so that we don't keep downloading the same everytime the kernel starts
resources = ['punkt', 'punkt_tab', 'stopwords']

for res in resources:
    try:
        nltk.data.find(f'tokenizers/{res}' if 'punkt' in res else f'corpora/{res}')
    except LookupError:
        nltk.download(res)

In [32]:
# Loading the standard list of English stop words from the NLTK
stop_words = set(stopwords.words('english'))

def clean_storyline(text):
    """
    Cleans, tokenizes, and removes stop words from a given text string.
    """
    # Handle any empty cells or NaN values in the dataset
    if not isinstance(text, str):
        return ""
        
    # Step 1: Lowercase the text
    text = text.lower()
    
    # Step 2: Remove punctuation and special characters
    # ^a-zA-Z\s means: "Keep only letters (a-z, A-Z) and spaces (\s)"
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Step 3: Tokenize the text (split the sentence into individual words)
    tokens = word_tokenize(text)
    
    # Step 4: Remove stop words (like 'the', 'a', 'is', 'on')
    # and remove any extra blank spaces
    cleaned_tokens = [word for word in tokens if word not in stop_words and word.strip() != '']
    
    # Step 5: Join the tokens back into a single string separated by spaces
    # TF-IDF expects a single string per document, not a list of words
    return ' '.join(cleaned_tokens)

# Apply the function to create your new column
df['Clean_Storyline'] = df['Storyline'].apply(clean_storyline)

In [31]:
df[['Storyline', 'Clean_Storyline']].head()

,Storyline,Clean_Storyline
0,"Paul Atreides unites with the Fremen while on a warpath of revenge against the conspirators who destroyed his family. Facing a choice between the love of his life and the fate of the universe, he endeavors to prevent a terrible future.",paul atreides unites fremen warpath revenge conspirators destroyed family facing choice love life fate universe endeavors prevent terrible future
1,"A life-affirming, genre-bending story about three chapters in the life of an ordinary man named Charles Krantz.",lifeaffirming genrebending story three chapters life ordinary man named charles krantz
2,"A fading celebrity takes a black-market drug: a cell-replicating substance that helps her create a younger, better version of herself.",fading celebrity takes blackmarket drug cellreplicating substance helps create younger better version
3,"A stuntman, fresh off an almost career-ending accident, has to track down a missing movie star, solve a conspiracy and try to win back the love of his life while still doing his day job.",stuntman fresh almost careerending accident track missing movie star solve conspiracy try win back love life still day job
4,"While scavenging the deep ends of a derelict space station, a group of young space colonists come face to face with the most terrifying life form in the universe.",scavenging deep ends derelict space station group young space colonists come face face terrifying life form universe


TF-IDF Vectorizer

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

temp_tfidf = TfidfVectorizer()
temp_tfidf.fit_transform(df['Clean_Storyline'])
print(len(temp_tfidf.get_feature_names_out()))

11356


In [35]:

# Using min_df=2 means: "Ignore words that appear in only 1 movie storyline"
tfidf = TfidfVectorizer(min_df=2, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Clean_Storyline'])

print(f"New vocabulary size: {len(tfidf.get_feature_names_out())}")

New vocabulary size: 4685


In [36]:
# 3. Check the shape of your new matrix
print(f"Dataset shape: {df.shape}")
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
# Expected output: (2500, 5000) -> 2500 movies, 5000 unique words

# Optional: Look at the actual words the model learned
feature_names = tfidf.get_feature_names_out()
print(f"Sample vocabulary: {feature_names[100:110]}")

Dataset shape: (2455, 3)
TF-IDF Matrix shape: (2455, 4685)
Sample vocabulary: ['agnes' 'ago' 'agrees' 'ah' 'ahead' 'ai' 'aid' 'aided' 'ailing' 'aiming']


In [38]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Calculate the Similarity Matrix
# Assuming 'tfidf_matrix' is the output from your previous TF-IDF step
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Similarity Matrix Shape: {cosine_sim.shape}")
# This will output (2455, 2455). Every movie is compared to every other movie.

# 2. Build the Recommendation Function
# Create a reverse mapping of movie titles to indices for quick lookups
indices = pd.Series(df.index, index=df['Movie_Name']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim):
    # Check if movie exists in dataset
    if title not in indices:
        return "Movie not found. Please check the spelling."
        
    # Get the index of the movie that matches the title
    idx = indices[title]

    # Get the pairwise similarity scores of all movies with that movie
    # list(enumerate()) keeps track of the original index while we sort
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 5 most similar movies
    # We slice [1:6] because index 0 is the movie itself (score of 1.0)
    sim_scores = sim_scores[1:6]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 5 most similar movies along with their storylines
    return df[['Movie_Name', 'Storyline']].iloc[movie_indices]

# 3. Test your system!
# Replace this with an actual movie name from your CSV
print(get_recommendations('Dune: Part Two'))

Similarity Matrix Shape: (2455, 2455)
                     Movie_Name  \
1807                      Obraz   
867   Terminator: Skynet Rising   
11         Deadpool & Wolverine   
1132                 Der Vierer   
1227           Bloodline Killer   

                                                                                                                                                                                                                                           Storyline  
1807                                                                                               A hunted child will find refuge in the house of the enemy. The host faces a terrible choice: save the child's life or risk losing his own family.  
867                                                                                 Since Legion was destroyed, the machines have been working to restore Skynet. Now the resistance has discovered their plans and must find a way to prevent them.  
11        

In [39]:
import numpy as np

def get_top_5_recommendations(user_vector, tfidf_matrix, df):
    """
    Ranks movies by comparing a user input vector to the existing movie matrix.
    """
    # 1. Calculate cosine similarity between input and all movies
    # This results in a vector of 2455 scores
    from sklearn.metrics.pairwise import cosine_similarity
    sim_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

    # 2. Pair each score with its index using enumerate
    # (index, score)
    indexed_scores = list(enumerate(sim_scores))

    # 3. Sort by the score (element at index 1) in descending order
    ranked_scores = sorted(indexed_scores, key=lambda x: x[1], reverse=True)

    # 4. Select the top 5 (skip the first one if the input was an existing movie)
    # Usually, we take [1:6] if comparing a movie to itself, or [0:5] for a new query
    top_indices = [i[0] for i in ranked_scores[1:6]]
    top_scores = [i[1] for i in ranked_scores[1:6]]

    # 5. Retrieve the names and storylines from the original DataFrame
    recommendations = df.iloc[top_indices].copy()
    recommendations['Match_Score'] = top_scores
    
    return recommendations[['Movie_Name', 'Match_Score', 'Storyline']]